## **Libaries Installed**

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries loaded!")

Libraries loaded!


## Load Datasets

In [5]:
import os

# Load datasets using relative paths
results = pd.read_csv(os.path.join('Data', 'results.csv'))
goalscorers = pd.read_csv(os.path.join('Data', 'goalscorers.csv'))
shootouts = pd.read_csv(os.path.join('Data', 'shootouts.csv'))

print("Results shape:", results.shape)
print("Goalscorers shape:", goalscorers.shape)
print("Shootouts shape:", shootouts.shape)

Results shape: (49437, 9)
Goalscorers shape: (47601, 8)
Shootouts shape: (678, 5)


## Preview Data

In [7]:
results.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


## Filter Data (2018-2026)

In [8]:
# Convert date and filter
results['date'] = pd.to_datetime(results['date'])
results = results[results['date'] >= '2018-01-01']

print("Filtered shape:", results.shape)
print("\nData types:")
print(results.dtypes)
print("\nMissing values:")
print(results.isnull().sum())

Filtered shape: (8140, 9)

Data types:
date          datetime64[us]
home_team                str
away_team                str
home_score           float64
away_score           float64
tournament               str
city                     str
country                  str
neutral                 bool
dtype: object

Missing values:
date           0
home_team      0
away_team      0
home_score    72
away_score    72
tournament     0
city           0
country        0
neutral        0
dtype: int64


## Clean Missing Values

In [9]:
# Drop rows where scores are missing
results = results.dropna(subset=['home_score', 'away_score'])

print("Shape after dropping nulls:", results.shape)
print("Missing values remaining:", results.isnull().sum().sum())

Shape after dropping nulls: (8068, 9)
Missing values remaining: 0


## Add Target Variable

In [10]:
# Create result column: 1 = home win, 0 = draw, -1 = away win
def get_result(row):
    if row['home_score'] > row['away_score']:
        return 1
    elif row['home_score'] < row['away_score']:
        return -1
    else:
        return 0

results['result'] = results.apply(get_result, axis=1)

print(results['result'].value_counts())

result
 1    3853
-1    2354
 0    1861
Name: count, dtype: int64


## Add Tournament Weights

In [11]:
# Weight matches by importance
tournament_weights = {
    'FIFA World Cup': 3.0,
    'FIFA World Cup qualification': 2.0,
    'UEFA Euro': 2.0,
    'UEFA Euro qualification': 1.5,
    'Copa América': 2.0,
    'Africa Cup of Nations': 2.0,
    'Friendly': 0.5,
    'Confederations Cup': 1.5
}

# Apply weights (default 1.0 for unlisted tournaments)
results['weight'] = results['tournament'].map(tournament_weights).fillna(1.0)

print(results['weight'].value_counts())

weight
1.0    3247
0.5    2237
2.0    1955
1.5     501
3.0     128
Name: count, dtype: int64


## Save Cleaned Data

In [12]:
results.to_csv(os.path.join('Cleaned_Data', 'results_cleaned.csv'), index=False)
print("Saved successfully!")
print("Final shape:", results.shape)

Saved successfully!
Final shape: (8068, 11)
